In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import warnings
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score

import os

# Additional Imports for Hyperparameter Tuning
import optuna
from imblearn.pipeline import Pipeline as ImbPipeline

# Ensure reproducibility
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

# Limit each parallel process to one thread per library
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['TORCH_NUM_THREADS'] = '1'  # For PyTorch

warnings.filterwarnings("ignore")  # Optional: Suppress warnings for cleaner output

# ----------------------------
# Data Preparation
# ----------------------------

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")

# Define feature groups as per user
feature_groups = {
    'Genotype': [
        'rs11225395', 'rs1144393', 'rs650108', 'rs591058', 'rs2252070', 'rs4986938', 'rs1800012', 'rs4789932', 'rs9340799', 'rs970547', 
        'rs1800795', 'rs13946', 'rs12722', 'class1_SNP_risk_score', 'rs7528684', 'rs4919510', 'rs1937810', 'rs6481512', 'rs1249269', 
        'rs12574452', 'rs12429486', 'rs4454832', 'rs2761884', 'rs62051384', 'rs4362400', 'rs2586488', 'rs2277698', 'rs1045485', 
        'rs143383', 'rs17576', 'rs2305948', 'rs1011814', 'rs11154027', 'rs2234693', 'rs1643821', 'rs2010963', 'rs10263021', 'rs149047058', 
        'rs420257', 'rs42517', 'rs42522', 'rs42531', 'rs413826', 'rs2104772', 'rs1330363', 'class12_SNP_risk_score', 'rs3753841', 
        'rs57104447', 'rs1887632', 'rs4654760', 'rs1137101', 'rs2306033', 'rs2277268', 'rs4988321', 'rs11232681', 'rs1718119', 'rs3751143', 
        'rs1544410', 'rs2228570', 'rs4328262', 'rs1021188', 'rs74544784', 'rs78391032', 'rs77569527', 'rs117544024', 'rs912336', 
        'rs3218791', 'rs911263', 'rs2525504', 'rs17756404', 'rs4903399', 'rs10132091', 'rs17583842', 'rs1676303', 'rs11629171', 
        'rs2281518', 'rs2285053', 'rs71404070', 'rs710079', 'rs2858056', 'rs820218', 'rs3018362', 'rs1800470', 'rs1800469', 'rs25487', 
        'rs25489', 'rs2289360', 'rs183364169', 'rs11177', 'rs6617', 'rs3219008', 'rs13107325', 'rs60713544', 'rs145648292', 'rs4244032', 
        'rs12656106', 'rs3045', 'rs187483', 'rs4701616', 'rs144414988', 'rs1800629', 'rs10484958', 'rs4730153', 'rs1800797', 'rs1554606', 
        'rs2237352', 'rs4725069', 'rs12154667', 'rs1548456', 'rs3216902', 'rs35360670', 'rs13317', 'rs1800972', 'rs7035322', 'rs7021589', 
        'rs72758637', 'rs10759753', 'rs3789870', 'rs1138545', 'rs3196378', 'rs1134170', 'rs10992075', 'rs1590', 'rs144371252', 
        'rs761804508', 'class123_SNP_risk_score', 'sex'
    ],
    'History': [
        'Age', 'lower_limb_days_total', 'average_run_hours', 'average_interval_training_frequency', 'EDEQ_total', 'tracking_period_injury',
        'past_stress_injury', 'LEAF-Q', 'Athlete_Score', 'average_run_frequency', 'past_month_injury'
    ],
    'Phenotype': [
        'hip_abduction_peak_torque', 'total_ad_ab_ratio', 'knee_extension_peak_torque', 'knee_flexion_peak_torque', 'navicular_drop', 
        'navicular_drop_asymmetry', 'Q_angle', 'Q_angle_asymmetry', 'VALR_12', 'Impact_peak_12', 'Duty_factor_12', 'BMI', 'BMD_spine',
        'hip_abduction_peak_torque_asymmetry', 'hip_adduction_peak_torque', 'hip_adduction_peak_torque_asymmetry', 
        'knee_extension_peak_torque_asymmetry', 'knee_flexion_peak_torque_asymmetry', 'total_fl_ex_ratio', 'leg_lean_mass', 
        'hip_abduction_peak_angle', 'hip_abduction_peak_angle_asymmetry', 'hip_adduction_peak_angle', 'hip_adduction_peak_angle_asymmetry', 
        'ad_ab_ratio_asymmetry', 'knee_extension_peak_angle', 'knee_extension_peak_angle_asymmetry', 'knee_flexion_peak_angle', 
        'knee_flexion_peak_angle_asymmetry', 'fl_ex_ratio_asymmetry', 'VILR_10', 'VALR_10', 'VILR_asymmetry_10', 'VALR_asymmetry_10', 
        'Impact_peak_10', 'Impact_peak_asymmetry_10', 'Flight_time_10', 'Contact_time_10', 'Duty_factor_10', 'Step_frequency_10', 
        'Cadence_asymmetry_10', 'Duty_factor_asymmetry_10', 'VILR_12', 'VILR_asymmetry_12', 'VALR_asymmetry_12', 
        'Impact_peak_asymmetry_12', 'Flight_time_12', 'Contact_time_12', 'Step_frequency_12', 'Cadence_asymmetry_12', 
        'Duty_factor_asymmetry_12', 'Alt_strike', 'height', 'Mass', 'thigh_lean_mass', 'thigh_ffmi', 'lower_leg_lean_mass', 
        'lower_leg_ffmi', 'leg_ffmi', 'total_lean_mass', 'total_ffmi', 'calf_size', 'BMD_hip', 'BMD_body',
    ],
    'Behaviour': [
        'fat_intake_avg', 'past_month_distance', 'past_month_ratio', 'SC_past_season', 'non_running_past_season', 'fat_intake_BW', 
        'fat_percentage_avg', 'average_energy_availability', 'protein_intake_BW', 'omega3_intake_BW', 'vitaminD_intake_BW', 
        'vitaminC_intake_BW', 'vitaminE_intake_BW', 'calcium_intake_BW', 'copper_intake_BW', 'iron_intake_BW', 'glycine_intake_BW', 
        'arginine_intake_BW', 'past_month_min', 'past_week_ratio', 'past_month_volume_low', 'past_week_ratio_low', 'past_month_ratio_low', 
        'past_month_volume_moderate', 'past_week_ratio_moderate', 'past_month_ratio_moderate', 'past_month_volume_high', 
        'past_week_ratio_high', 'past_month_ratio_high', 'past_month_volume_very_high', 'past_week_ratio_very_high', 
        'past_month_ratio_very_high', 'past_month_calculated_volume', 'past_week_ratio_calculated_volume', 
        'past_month_ratio_calculated_volume', 'resistance_training_past_month', 'resistance_training_past_season', 
        'bodyweight_exercises_past_month', 'bodyweight_exercises_past_season', 'core_stability_past_month', 'core_stability_past_season', 
        'balance_training_past_month', 'balance_training_past_season', 'plyometrics_past_month', 'plyometrics_past_season', 
        'drills_past_month', 'drills_past_season', 'circuit_training_past_month', 'circuit_training_past_season', 'barefoot_past_month', 
        'barefoot_past_season', 'stretching_past_month', 'stretching_past_season', 'SC_past_month', 'non_running_past_month'
    ]
}

# Assuming 'data' and 'feature_groups' are already defined
# Define predictors and outcome
X = data.drop(columns=['RRI'])  # Predictors
y = data['RRI']  # Outcome

# Ensure that the feature groups exist in the dataset
for group in feature_groups:
    feature_groups[group] = [feature for feature in feature_groups[group] if feature in X.columns]

# Initialize Logistic Regression with L1 penalty
# Choose 'saga' solver as it supports L1 regularization and is suitable for large datasets
logistic = LogisticRegression(penalty='l1', solver='saga', C=1.0, max_iter=10000, n_jobs=-1)

# Fit the model
logistic.fit(X, y)

# Get the coefficients and map them to feature names
coefficients = pd.Series(logistic.coef_[0], index=X.columns)

# Select features with non-zero coefficients
selected_features = coefficients[coefficients != 0].abs().sort_values(ascending=False).index.tolist()

print(selected_features)
print(f"Selected Features ({len(selected_features)}):")
print(coefficients.loc[selected_features])

# Update feature groups based on selected features, preserving Logistic Lasso ranking
selected_feature_groups = {group: [] for group in feature_groups}

for feature in selected_features:
    for group in feature_groups:
        if feature in feature_groups[group]:
            selected_feature_groups[group].append(feature)
            break  # Assuming each feature belongs to only one group

print("Selected Feature Groups:")
for group, features in selected_feature_groups.items():
    print(f"{group} ({len(features)}): {features}")

# Now, redefine X based on selected features
X_selected = X[selected_features].copy()

# Split feature groups
X_genotype = X_selected[selected_feature_groups['Genotype']].values.astype(np.float32)
X_history = X_selected[selected_feature_groups['History']].values.astype(np.float32)
X_phenotype = X_selected[selected_feature_groups['Phenotype']].values.astype(np.float32)
X_behaviour = X_selected[selected_feature_groups['Behaviour']].values.astype(np.float32)

# Convert target to tensor (assuming you are using PyTorch)
y_tensor = torch.tensor(y.values.astype(np.float32))

# ----------------------------
# Dataset and DataLoader
# ----------------------------

class CustomDataset(Dataset):
    def __init__(self, genotype, history, phenotype, behaviour, labels):
        self.genotype = genotype
        self.history = history
        self.phenotype = phenotype
        self.behaviour = behaviour
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.genotype[idx],
            self.history[idx],
            self.phenotype[idx],
            self.behaviour[idx],
            self.labels[idx]
        )

# ----------------------------
# FeatureAttention and MaskedLinear Layers
# ----------------------------

class FeatureAttention(nn.Module):
    def __init__(self, feature_dim):
        super(FeatureAttention, self).__init__()
        self.attention = nn.Sequential(
            nn.Linear(feature_dim, max(1, feature_dim // 2)),
            nn.ReLU(),
            nn.Linear(max(1, feature_dim // 2), feature_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        weights = self.attention(x)
        return x * weights

class MaskedLinear(nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super(MaskedLinear, self).__init__()
        # Initialize the linear layer
        self.linear = nn.Linear(in_features, out_features, bias)
        # Initialize mask parameters with the same shape as weights
        self.mask_param = nn.Parameter(torch.ones_like(self.linear.weight))
        if bias:
            self.bias = self.linear.bias
        else:
            self.register_parameter('bias', None)
    
    def forward(self, x):
        # Apply sigmoid to mask parameters to get gating probabilities
        mask = torch.sigmoid(self.mask_param)
        # Apply the mask to the weights
        masked_weight = self.linear.weight * mask
        return F.linear(x, masked_weight, self.bias)
    
    def get_binary_mask(self, threshold=0.5):
        """
        Returns a binary mask based on the gating parameters and a specified threshold.
        """
        with torch.no_grad():
            mask = torch.sigmoid(self.mask_param)
            binary_mask = (mask > threshold).float()
        return binary_mask

# ----------------------------
# Modified Model Definition with Optional Attention and Mask
# ----------------------------

class CustomMLPWithOptionalComponents(nn.Module):
    def __init__(self, genotype_size, history_size, phenotype_size, behaviour_size,
                 use_attention=True, use_mask=True):
        super(CustomMLPWithOptionalComponents, self).__init__()
        
        self.use_attention = use_attention
        self.use_mask = use_mask

        # Attention for Genotype Features
        if self.use_attention:
            self.attention_genotype = FeatureAttention(genotype_size)
        
        # Define the main layers with MaskedLinear or Linear based on use_mask
        LinearLayer = MaskedLinear if self.use_mask else nn.Linear

        # Genotype to History
        self.genotype_to_history = LinearLayer(genotype_size, history_size, bias=False)
        self.bn_genotype_to_history = nn.BatchNorm1d(history_size)
        self.genotype_to_history_bias = nn.Parameter(torch.zeros(history_size))
        if self.use_attention:
            self.attention_history = FeatureAttention(history_size + 1)  # +1 for extra input
        
        # History to Phenotype
        self.history_to_phenotype = LinearLayer(history_size + 1, phenotype_size, bias=False)
        self.bn_history_to_phenotype = nn.BatchNorm1d(phenotype_size)
        self.history_to_phenotype_bias = nn.Parameter(torch.zeros(phenotype_size))
        if self.use_attention:
            self.attention_phenotype = FeatureAttention(phenotype_size + 1)
        
        # Phenotype to Behaviour
        self.phenotype_to_behaviour = LinearLayer(phenotype_size + 1, behaviour_size, bias=False)
        self.bn_phenotype_to_behaviour = nn.BatchNorm1d(behaviour_size)
        self.phenotype_to_behaviour_bias = nn.Parameter(torch.zeros(behaviour_size))
        if self.use_attention:
            self.attention_behaviour = FeatureAttention(behaviour_size + 1)
        
        # Behaviour to Output
        self.behaviour_to_output = LinearLayer(behaviour_size + 1, 1)
        
        # Extra regular node layers (no batch norm needed)
        self.genotype_to_history_extra = LinearLayer(genotype_size, 1, bias=True)
        self.history_to_phenotype_extra = LinearLayer(history_size + 1, 1, bias=True)
        self.phenotype_to_behaviour_extra = LinearLayer(phenotype_size + 1, 1, bias=True)
        
        # Activation function
        self.relu = torch.nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
        # Initialize weights using Xavier/Glorot initialization
        for layer in [
            self.genotype_to_history, 
            self.history_to_phenotype, 
            self.phenotype_to_behaviour, 
            self.behaviour_to_output,
            self.genotype_to_history_extra,
            self.history_to_phenotype_extra,
            self.phenotype_to_behaviour_extra
        ]:
            if self.use_mask:
                nn.init.xavier_uniform_(layer.linear.weight)
                if layer.linear.bias is not None:
                    nn.init.zeros_(layer.linear.bias)
            else:
                nn.init.xavier_uniform_(layer.weight)
                if layer.bias is not None:
                    nn.init.zeros_(layer.bias)
    
    def forward(self, genotype, history, phenotype, behaviour):
        # Genotype to History with optional attention
        if self.use_attention:
            genotype_att = self.attention_genotype(genotype)
        else:
            genotype_att = genotype
        main_history_input = self.genotype_to_history(genotype_att) * history
        main_history_input = self.bn_genotype_to_history(main_history_input)
        main_history_input = main_history_input + self.genotype_to_history_bias
        
        # Extra input (no batch norm)
        extra_history_input = self.genotype_to_history_extra(genotype_att)
        combined_history_input = torch.cat([main_history_input, extra_history_input], dim=1)
        history_output = self.relu(combined_history_input)
        
        # History to Phenotype with optional attention
        if self.use_attention:
            history_att = self.attention_history(history_output)
        else:
            history_att = history_output
        main_phenotype_input = self.history_to_phenotype(history_att) * phenotype
        main_phenotype_input = self.bn_history_to_phenotype(main_phenotype_input)
        main_phenotype_input = main_phenotype_input + self.history_to_phenotype_bias
        
        # Extra input for phenotype (no batch norm)
        extra_phenotype_input = self.history_to_phenotype_extra(history_att)
        combined_phenotype_input = torch.cat([main_phenotype_input, extra_phenotype_input], dim=1)
        phenotype_output = self.relu(combined_phenotype_input) 
        
        # Phenotype to Behaviour with optional attention
        if self.use_attention:
            phenotype_att = self.attention_phenotype(phenotype_output)
        else:
            phenotype_att = phenotype_output
        main_behaviour_input = self.phenotype_to_behaviour(phenotype_att) * behaviour
        main_behaviour_input = self.bn_phenotype_to_behaviour(main_behaviour_input)
        main_behaviour_input = main_behaviour_input + self.phenotype_to_behaviour_bias

        # Extra input for behaviour (no batch norm)
        extra_behaviour_input = self.phenotype_to_behaviour_extra(phenotype_att)
        combined_behaviour_input = torch.cat([main_behaviour_input, extra_behaviour_input], dim=1)
        behaviour_output = self.relu(combined_behaviour_input) 
        
        # Behaviour to Output with optional attention
        if self.use_attention:
            behaviour_att = self.attention_behaviour(behaviour_output)
        else:
            behaviour_att = behaviour_output
        final_input = self.behaviour_to_output(behaviour_att)
        final_output = self.sigmoid(final_input)
        
        return final_output.squeeze()  # Return as (batch_size,)

# ----------------------------
# Hyperparameter Tuning with Optuna
# ----------------------------

# Define the objective function
def objective(trial):
    # Hyperparameters to tune
    # Number of features per group
    n_genotype = trial.suggest_int('n_genotype', 1, len(selected_feature_groups['Genotype']))
    n_history = trial.suggest_int('n_history', 1, len(selected_feature_groups['History']))
    n_phenotype = trial.suggest_int('n_phenotype', 1, len(selected_feature_groups['Phenotype']))
    n_behaviour = trial.suggest_int('n_behaviour', 1, len(selected_feature_groups['Behaviour']))
    
    # Learning rate
    lr = trial.suggest_loguniform('learning_rate', 1e-5, 1e-2)
    
    # Number of epochs
    epochs = trial.suggest_int('epochs', 500, 3000)
    
    # Batch size
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128, 256, 512])
    
    # Whether to use attention and mask
    use_attention = True
    use_mask = False
    
    # Select features based on the number of features per group
    selected_genotype_features = selected_feature_groups['Genotype'][:n_genotype]
    selected_history_features = selected_feature_groups['History'][:n_history]
    selected_phenotype_features = selected_feature_groups['Phenotype'][:n_phenotype]
    selected_behaviour_features = selected_feature_groups['Behaviour'][:n_behaviour]
    
    # Combine selected features
    current_selected_features = selected_genotype_features + selected_history_features + \
                                 selected_phenotype_features + selected_behaviour_features

    print(current_selected_features)
    
    # Prepare data based on current_selected_features
    X_current = X[current_selected_features].copy()
    
    # Update feature groups
    current_feature_groups = {
        'Genotype': selected_genotype_features,
        'History': selected_history_features,
        'Phenotype': selected_phenotype_features,
        'Behaviour': selected_behaviour_features
    }
    
    # Split feature groups
    X_genotype_current = X_current[current_feature_groups['Genotype']].values.astype(np.float32)
    X_history_current = X_current[current_feature_groups['History']].values.astype(np.float32)
    X_phenotype_current = X_current[current_feature_groups['Phenotype']].values.astype(np.float32)
    X_behaviour_current = X_current[current_feature_groups['Behaviour']].values.astype(np.float32)
    
    # Convert target to tensor
    y_current = y.values.astype(np.float32)  # Convert to NumPy array
    
    # Stratified K-Fold Cross Validation with 10 folds
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    
    # Initialize lists to store metrics
    auc_scores = []
    f1_scores = []
    accuracy_scores = []
    precision_scores = []
    
    # Define the fold processing function
    def train_evaluate_fold(train_index, valid_index):
        # Split the data
        X_train_gen, X_valid_gen = X_genotype_current[train_index], X_genotype_current[valid_index]
        X_train_hist, X_valid_hist = X_history_current[train_index], X_history_current[valid_index]
        X_train_pheno, X_valid_pheno = X_phenotype_current[train_index], X_phenotype_current[valid_index]
        X_train_behav, X_valid_behav = X_behaviour_current[train_index], X_behaviour_current[valid_index]
        y_train_fold, y_valid_fold = y_current[train_index], y_current[valid_index]
        
        # Handle class imbalance using SMOTE (you can choose other methods)
        X_train_combined = np.hstack((X_train_gen, X_train_hist, X_train_pheno, X_train_behav))
        X_train_res, y_train_res = (X_train_combined, y_train_fold)  # Placeholder for SMOTE
        
        # After resampling, split back into feature groups
        n_gen = X_train_gen.shape[1]
        n_hist = X_train_hist.shape[1]
        n_pheno = X_train_pheno.shape[1]
        n_behav = X_train_behav.shape[1]
        
        X_train_gen_res = X_train_res[:, :n_gen]
        X_train_hist_res = X_train_res[:, n_gen:n_gen+n_hist]
        X_train_pheno_res = X_train_res[:, n_gen+n_hist:n_gen+n_hist+n_pheno]
        X_train_behav_res = X_train_res[:, n_gen+n_hist+n_pheno:]
        
        # Create datasets and dataloaders
        train_dataset = CustomDataset(
            genotype=X_train_gen_res,
            history=X_train_hist_res,
            phenotype=X_train_pheno_res,
            behaviour=X_train_behav_res,
            labels=y_train_res
        )
        
        valid_dataset = CustomDataset(
            genotype=X_valid_gen,
            history=X_valid_hist,
            phenotype=X_valid_pheno,
            behaviour=X_valid_behav,
            labels=y_valid_fold
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
        
        # Initialize the model
        model = CustomMLPWithOptionalComponents(
            genotype_size=X_train_gen_res.shape[1],
            history_size=X_train_hist_res.shape[1],
            phenotype_size=X_train_pheno_res.shape[1],
            behaviour_size=X_train_behav_res.shape[1],
            use_attention=use_attention,
            use_mask=use_mask
        ).to(device)
        
        # Define optimizer and loss function
        optimizer = optim.Adam(model.parameters(), lr=lr)
        criterion = nn.BCELoss()
        
        # Training loop
        model.train()
        for epoch in range(epochs):
            for batch in train_loader:
                genotype, history, phenotype, behaviour, labels = batch
                genotype = genotype.to(device)
                history = history.to(device)
                phenotype = phenotype.to(device)
                behaviour = behaviour.to(device)
                labels = labels.to(device)
                
                optimizer.zero_grad()
                outputs = model(genotype, history, phenotype, behaviour)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
        
        # Evaluation
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for batch in valid_loader:
                genotype, history, phenotype, behaviour, labels = batch
                genotype = genotype.to(device)
                history = history.to(device)
                phenotype = phenotype.to(device)
                behaviour = behaviour.to(device)
                
                outputs = model(genotype, history, phenotype, behaviour)
                all_preds.extend(outputs.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        # Compute Metrics
        auc = roc_auc_score(all_labels, all_preds)
        # Binarize predictions with a threshold of 0.5
        binarized_preds = [1 if p >= 0.5 else 0 for p in all_preds]
        f1 = f1_score(all_labels, binarized_preds)
        accuracy = accuracy_score(all_labels, binarized_preds)
        precision = precision_score(all_labels, binarized_preds)
        
        return auc, f1, accuracy, precision
    
    # Parallelize the fold processing
    results = Parallel(n_jobs=10)(
        delayed(train_evaluate_fold)(train_idx, valid_idx) for train_idx, valid_idx in skf.split(X_current, y_current)
    )
    
    # Unpack the results
    for auc, f1, accuracy, precision in results:
        auc_scores.append(auc)
        f1_scores.append(f1)
        accuracy_scores.append(accuracy)
        precision_scores.append(precision)
    
    # Calculate average and standard deviation for each metric
    avg_auc = np.mean(auc_scores)
    std_auc = np.std(auc_scores)
    
    avg_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)
    
    avg_accuracy = np.mean(accuracy_scores)
    std_accuracy = np.std(accuracy_scores)
    
    avg_precision = np.mean(precision_scores)
    std_precision = np.std(precision_scores)
    
    # Log the metrics to Optuna's trial
    trial.set_user_attr("f1_score", avg_f1)
    trial.set_user_attr("f1_score_std", std_f1)
    trial.set_user_attr("accuracy", avg_accuracy)
    trial.set_user_attr("accuracy_std", std_accuracy)
    trial.set_user_attr("precision", avg_precision)
    trial.set_user_attr("precision_std", std_precision)
    
    # Optionally, print the metrics for each trial
    print(f"Trial {trial.number}:")
    print(f"  AUC: {avg_auc:.4f} (±{std_auc:.4f})")
    print(f"  F1 Score: {avg_f1:.4f} (±{std_f1:.4f})")
    print(f"  Accuracy: {avg_accuracy:.4f} (±{std_accuracy:.4f})")
    print(f"  Precision: {avg_precision:.4f} (±{std_precision:.4f})")
    print("-" * 30)
    
    # Return the average AUC as the objective to maximize
    return avg_auc

# Set device
device = torch.device('cpu')

# Create the Optuna study
study = optuna.create_study(direction='maximize')

# Optimize
study.optimize(objective, n_trials=500, timeout=None)  # Adjust n_trials and timeout as needed

# Function to retrieve and print metrics from the study
def print_study_results(study):
    print("Best Trial:")
    trial = study.best_trial
    
    print(f"  AUC: {trial.value:.4f}")
    print("  F1 Score: {:.4f} (Std: {:.4f})".format(
        trial.user_attrs.get("f1_score", np.nan),
        trial.user_attrs.get("f1_score_std", np.nan)
    ))
    print("  Accuracy: {:.4f} (Std: {:.4f})".format(
        trial.user_attrs.get("accuracy", np.nan),
        trial.user_attrs.get("accuracy_std", np.nan)
    ))
    print("  Precision: {:.4f} (Std: {:.4f})".format(
        trial.user_attrs.get("precision", np.nan),
        trial.user_attrs.get("precision_std", np.nan)
    ))
    print("  Params: ")
    for key, value in trial.params.items():
        print(f"    {key}: {value}")

# Print the best trial's metrics
print_study_results(study)

[I 2024-11-12 13:11:07,507] A new study created in memory with name: no-name-21daf789-de9b-41d3-9d2b-a9e78b646a7e


['rs145648292', 'average_run_hours', 'past_month_ratio', 'past_week_ratio_low', 'rs1676303', 'vitaminD_intake_BW', 'knee_extension_peak_angle_asymmetry', 'rs3045', 'Q_angle_asymmetry', 'hip_abduction_peak_angle', 'rs1144393', 'ad_ab_ratio_asymmetry', 'resistance_training_past_season', 'rs2277268', 'class1_SNP_risk_score', 'BMD_body', 'Duty_factor_asymmetry_10', 'rs2104772', 'rs7035322', 'past_month_distance', 'glycine_intake_BW', 'past_month_ratio_calculated_volume', 'rs11225395', 'rs4701616', 'rs3218791', 'rs13107325', 'Alt_strike', 'rs4988321', 'tracking_period_injury', 'rs1590', 'fl_ex_ratio_asymmetry', 'rs1800629', 'rs10484958', 'rs2306033', 'rs10132091', 'leg_ffmi', 'Impact_peak_asymmetry_12', 'calcium_intake_BW', 'rs2289360', 'past_month_injury', 'rs1800972', 'Q_angle', 'average_energy_availability', 'rs912336', 'rs4919510', 'rs6481512', 'rs25489', 'rs1249269', 'knee_flexion_peak_angle_asymmetry', 'rs2234693', 'hip_adduction_peak_angle_asymmetry', 'fat_percentage_avg', 'rs7839103

[I 2024-11-12 13:15:33,793] Trial 0 finished with value: 0.6876724148186294 and parameters: {'n_genotype': 73, 'n_history': 4, 'n_phenotype': 4, 'n_behaviour': 11, 'learning_rate': 0.007167953493066214, 'epochs': 2168, 'batch_size': 512}. Best is trial 0 with value: 0.6876724148186294.


Trial 0:
  AUC: 0.6877 (±0.0421)
  F1 Score: 0.1881 (±0.0662)
  Accuracy: 0.8963 (±0.0135)
  Precision: 0.3600 (±0.1607)
------------------------------
['rs145648292', 'rs1676303', 'rs3045', 'rs1144393', 'rs2277268', 'class1_SNP_risk_score', 'rs2104772', 'rs7035322', 'rs11225395', 'rs4701616', 'rs3218791', 'rs13107325', 'rs4988321', 'rs1590', 'rs1800629', 'rs10484958', 'rs2306033', 'rs10132091', 'rs2289360', 'rs1800972', 'rs912336', 'rs4919510', 'rs6481512', 'rs25489', 'rs1249269', 'rs2234693', 'rs78391032', 'rs2252070', 'rs2305948', 'rs3219008', 'rs2586488', 'rs1643821', 'rs591058', 'rs4903399', 'rs11232681', 'rs1800469', 'rs2281518', 'rs1134170', 'rs10992075', 'rs4454832', 'rs1544410', 'rs1887632', 'rs3018362', 'rs1021188', 'rs1800797', 'rs2285053', 'rs25487', 'rs4328262', 'rs3751143', 'rs62051384', 'rs2858056', 'rs12574452', 'rs11154027', 'rs4654760', 'rs420257', 'sex', 'rs1937810', 'rs144414988', 'rs7528684', 'rs970547', 'rs2237352', 'rs3753841', 'rs1800012', 'rs10263021', 'average

/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
[I 2024-11-12 13:30:47,860] Trial 1 finished with value: 0.7038087906670781 and parameters: {'n_genotype': 64, 'n_history': 1, 'n_phenotype': 1, 'n_behaviour': 3, 'learning_rate': 0.0008691360193373591, 'epochs': 950, 'batch_size': 32}. Best is trial 1 with value: 0.7038087906670781.


...

Trial 69:
  AUC: 0.7317 (±0.0427)
  F1 Score: 0.0364 (±0.0677)
  Accuracy: 0.9084 (±0.0023)
  Precision: 0.2571 (±0.3414)
------------------------------
['rs145648292', 'rs1676303', 'rs3045', 'rs1144393', 'rs2277268', 'class1_SNP_risk_score', 'rs2104772', 'rs7035322', 'rs11225395', 'rs4701616', 'rs3218791', 'rs13107325', 'rs4988321', 'rs1590', 'rs1800629', 'rs10484958', 'rs2306033', 'rs10132091', 'rs2289360', 'rs1800972', 'rs912336', 'rs4919510', 'rs6481512', 'rs25489', 'rs1249269', 'rs2234693', 'rs78391032', 'rs2252070', 'rs2305948', 'rs3219008', 'rs2586488', 'rs1643821', 'rs591058', 'rs4903399', 'rs11232681', 'rs1800469', 'rs2281518', 'rs1134170', 'rs10992075', 'rs4454832', 'rs1544410', 'rs1887632', 'rs3018362', 'rs1021188', 'rs1800797', 'rs2285053', 'rs25487', 'rs4328262', 'rs3751143', 'rs62051384', 'rs2858056', 'rs12574452', 'rs11154027', 'rs4654760', 'rs420257', 'sex', 'rs1937810', 'rs144414988', 'rs7528684', 'rs970547', 'rs2237352', 'rs3753841', 'rs1800012', 'average_run_hours', 

/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWar

Trial 70:
  AUC: 0.7223 (±0.0363)
  F1 Score: 0.0316 (±0.0525)
  Accuracy: 0.9071 (±0.0034)
  Precision: 0.2060 (±0.2456)
------------------------------
['rs145648292', 'rs1676303', 'rs3045', 'rs1144393', 'rs2277268', 'class1_SNP_risk_score', 'rs2104772', 'rs7035322', 'rs11225395', 'rs4701616', 'rs3218791', 'rs13107325', 'rs4988321', 'rs1590', 'rs1800629', 'rs10484958', 'rs2306033', 'rs10132091', 'rs2289360', 'rs1800972', 'rs912336', 'rs4919510', 'rs6481512', 'rs25489', 'rs1249269', 'rs2234693', 'rs78391032', 'rs2252070', 'rs2305948', 'rs3219008', 'rs2586488', 'rs1643821', 'rs591058', 'rs4903399', 'rs11232681', 'rs1800469', 'rs2281518', 'rs1134170', 'rs10992075', 'rs4454832', 'rs1544410', 'rs1887632', 'rs3018362', 'rs1021188', 'rs1800797', 'rs2285053', 'rs25487', 'rs4328262', 'rs3751143', 'rs62051384', 'rs2858056', 'rs12574452', 'rs11154027', 'rs4654760', 'rs420257', 'sex', 'rs1937810', 'rs144414988', 'rs7528684', 'rs970547', 'rs2237352', 'rs3753841', 'rs1800012', 'average_run_hours', 

/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/h_wu4/ml_env/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWar

Trial 71:
  AUC: 0.7088 (±0.0439)
  F1 Score: 0.0567 (±0.0613)
  Accuracy: 0.9071 (±0.0054)
  Precision: 0.2916 (±0.2713)
------------------------------
['rs145648292', 'rs1676303', 'rs3045', 'rs1144393', 'rs2277268', 'class1_SNP_risk_score', 'rs2104772', 'rs7035322', 'rs11225395', 'rs4701616', 'rs3218791', 'rs13107325', 'rs4988321', 'rs1590', 'rs1800629', 'rs10484958', 'rs2306033', 'rs10132091', 'rs2289360', 'rs1800972', 'rs912336', 'rs4919510', 'rs6481512', 'rs25489', 'rs1249269', 'rs2234693', 'rs78391032', 'rs2252070', 'rs2305948', 'rs3219008', 'rs2586488', 'rs1643821', 'rs591058', 'rs4903399', 'rs11232681', 'rs1800469', 'rs2281518', 'rs1134170', 'rs10992075', 'rs4454832', 'rs1544410', 'rs1887632', 'rs3018362', 'rs1021188', 'rs1800797', 'rs2285053', 'rs25487', 'rs4328262', 'rs3751143', 'rs62051384', 'rs2858056', 'rs12574452', 'rs11154027', 'rs4654760', 'rs420257', 'sex', 'rs1937810', 'rs144414988', 'rs7528684', 'rs970547', 'rs2237352', 'rs3753841', 'rs1800012', 'rs10263021', 'rs1757

In [7]:
print_study_results(study)

Best Trial:
  AUC: 0.7417
  F1 Score: 0.0456 (Std: 0.0528)
  Accuracy: 0.9075 (Std: 0.0046)
  Precision: 0.3000 (Std: 0.3253)
  Params: 
    n_genotype: 36
    n_history: 4
    n_phenotype: 8
    n_behaviour: 2
    learning_rate: 0.00013536078428508602
    epochs: 2045
    batch_size: 16
